In [1]:
import requests
import json
import time

# ============================================================
# CONFIG
# ============================================================

PROSPEO_API_KEY = "pk_afec0c742724b6790535cd64c7c2f138c66667650f7f7b3f6a583543c860d428"

SEARCH_URL = "https://api.prospeo.io/search-person"
ENRICH_URL = "https://api.prospeo.io/enrich-person"

HEADERS = {
    "X-KEY": PROSPEO_API_KEY,
    "Content-Type": "application/json"
}


# ============================================================
# SEARCH PEOPLE
# ============================================================

def search_people(
    job_titles,
    locations=None,
    companies=None,
    page=1
):
    """
    Search Prospeo for people matching your criteria.
    """

    filters = {
        "person_job_title": {
            "include": job_titles,
            "match_mode": "CONTAINS"
        }
    }

    # Optional location filter
    if locations:
        filters["person_location"] = {
            "include": locations
        }

    # Optional company websites
    if companies:
        filters["company"] = {
            "websites": {
                "include": companies
            }
        }

    payload = {
        "page": page,
        "filters": filters
    }

    print("\nSearching Prospeo...")
    print(json.dumps(payload, indent=2))

    response = requests.post(
        SEARCH_URL,
        headers=HEADERS,
        json=payload,
        timeout=30
    )

    print("\nHTTP status:", response.status_code)

    if not response.ok:
        print(response.text)
        return []

    data = response.json()

    if data.get("error"):
        print("Prospeo error:")
        print(data)
        return []

    results = data.get("results", [])

    print(f"\nFound {len(results)} people.")

    return results


# ============================================================
# ENRICH PERSON
# ============================================================

def enrich_person(person_id):
    """
    Enrich a Prospeo person using their person_id.
    Requests verified email only.
    """

    payload = {
        "only_verified_email": True,
        "data": {
            "person_id": person_id
        }
    }

    response = requests.post(
        ENRICH_URL,
        headers=HEADERS,
        json=payload,
        timeout=30
    )

    if response.status_code == 400:
        data = response.json()

        if data.get("error_code") == "NO_MATCH":
            return None

        print("Enrichment error:")
        print(data)
        return None

    if not response.ok:
        print("HTTP error:", response.status_code)
        print(response.text)
        return None

    data = response.json()

    if data.get("error"):
        print("Prospeo enrichment error:")
        print(data)
        return None

    return data


# ============================================================
# PROCESS PEOPLE
# ============================================================

def process_people(results):

    enriched_people = []

    total = len(results)

    for i, result in enumerate(results, start=1):

        person = result.get("person", {})
        company = result.get("company", {})

        person_id = person.get("person_id")

        print("\n" + "=" * 80)
        print(f"[{i}/{total}]")

        print(
            "Name:",
            person.get("full_name")
            or f"{person.get('first_name', '')} {person.get('last_name', '')}"
        )

        print("Title:", person.get("current_job_title"))
        print("Company:", company.get("name"))
        print("Location:", person.get("location"))

        if not person_id:
            print("No person ID.")
            continue

        print("Enriching...")

        enrichment = enrich_person(person_id)

        if not enrichment:
            print("No enrichment result.")
            continue

        enriched_person = enrichment.get("person", {})
        enriched_company = enrichment.get("company", {})

        record = {
            "person_id": enriched_person.get("person_id"),

            "first_name": enriched_person.get("first_name"),
            "last_name": enriched_person.get("last_name"),
            "full_name": enriched_person.get("full_name"),

            "job_title": enriched_person.get("current_job_title"),

            "linkedin_url": enriched_person.get("linkedin_url"),

            "email": enriched_person.get("email"),

            "email_status": enriched_person.get("email", {}).get("status")
            if isinstance(enriched_person.get("email"), dict)
            else None,

            "location": enriched_person.get("location"),

            "company": enriched_company.get("name"),
            "company_website": enriched_company.get("website"),

            # Keep everything returned by Prospeo
            "raw_person": enriched_person,
            "raw_company": enriched_company
        }

        enriched_people.append(record)

        print("Email:", record["email"])
        print("Email status:", record["email_status"])

        # Small delay
        time.sleep(0.2)

    return enriched_people


# ============================================================
# SAVE RESULTS
# ============================================================

def save_results(results, filename="prospeo_quant_people.json"):

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(
            results,
            f,
            indent=4,
            ensure_ascii=False
        )

    print("\nSaved:", filename)


# ============================================================
# MAIN
# ============================================================

def main():

    # Quant finance titles
    job_titles = [
        "Quantitative Researcher",
        "Quantitative Research Analyst",
        "Quantitative Trader",
        "Quantitative Analyst",
        "Systematic Trader",
        "Systematic Researcher",
        "Algorithmic Trader",
        "Algorithmic Researcher",
        "Portfolio Manager",
        "Systematic Portfolio Manager",
        "Quantitative Developer"
    ]

    # Example locations
    locations = [
        "London",
        "New York",
        "Chicago"
    ]

    # Search
    results = search_people(
        job_titles=job_titles,
        locations=locations,
        page=1
    )

    if not results:
        print("\nNo people found.")
        return

    # Enrich
    enriched_people = process_people(results)

    # Save
    save_results(enriched_people)

    print("\n" + "=" * 80)
    print("DONE")
    print("=" * 80)

    print(
        f"\nSuccessfully enriched {len(enriched_people)} people."
    )


if __name__ == "__main__":
    main()


Searching Prospeo...
{
  "page": 1,
  "filters": {
    "person_job_title": {
      "include": [
        "Quantitative Researcher",
        "Quantitative Research Analyst",
        "Quantitative Trader",
        "Quantitative Analyst",
        "Systematic Trader",
        "Systematic Researcher",
        "Algorithmic Trader",
        "Algorithmic Researcher",
        "Portfolio Manager",
        "Systematic Portfolio Manager",
        "Quantitative Developer"
      ],
      "match_mode": "CONTAINS"
    },
    "person_location": {
      "include": [
        "London",
        "New York",
        "Chicago"
      ]
    }
  }
}

HTTP status: 400
{"error":true,"error_code":"INVALID_FILTERS","filter_error":"Invalid value '{'include': ['London', 'New York', 'Chicago']}' for filter 'person_location'"}

No people found.


In [3]:
import requests
import json

# ============================================================
# CONFIG
# ============================================================

HEADERS = {
    "X-KEY": PROSPEO_API_KEY,
    "Content-Type": "application/json"
}


# ============================================================
# 1. FIND LOCATION
# ============================================================

def find_location(location):
    url = "https://api.prospeo.io/search-suggestions"

    payload = {
        "location_search": location
    }

    response = requests.post(
        url,
        headers=HEADERS,
        json=payload,
        timeout=30
    )

    print("Location search status:", response.status_code)

    if not response.ok:
        print(response.text)
        return None

    data = response.json()

    print("\nLocation suggestions:")
    print(json.dumps(data, indent=2, ensure_ascii=False))

    return data


# ============================================================
# 2. SEARCH PEOPLE
# ============================================================

def search_people(location):

    url = "https://api.prospeo.io/search-person"

    payload = {
        "page": 1,
        "filters": {

            "person_job_title": {
                "include": [
                    "Quantitative Researcher",
                    "Quantitative Research Analyst",
                    "Quantitative Trader",
                    "Quantitative Analyst",
                    "Systematic Trader",
                    "Systematic Researcher",
                    "Algorithmic Trader",
                    "Portfolio Manager",
                    "Quantitative Developer"
                ],
                "match_mode": "CONTAINS"
            },

            "person_location_search": {
                "include": [
                    location
                ]
            }
        }
    }

    print("\nSearching Prospeo...")
    print(json.dumps(payload, indent=2))

    response = requests.post(
        url,
        headers=HEADERS,
        json=payload,
        timeout=30
    )

    print("\nPeople search status:", response.status_code)

    if not response.ok:
        print(response.text)
        return []

    data = response.json()

    print("\nResults:")
    print(json.dumps(data, indent=2, ensure_ascii=False))

    return data


# ============================================================
# MAIN
# ============================================================

def main():

    location = input(
        "Enter location (example: London): "
    ).strip()

    if not location:
        print("Location cannot be empty.")
        return

    # First get Prospeo's location suggestions
    suggestions = find_location(location)

    if not suggestions:
        return

    print("\n----------------------------------------")
    print("IMPORTANT")
    print("----------------------------------------")
    print(
        "Look at the location suggestions above and "
        "copy the exact Prospeo location value."
    )

    exact_location = input(
        "\nPaste the exact Prospeo location here: "
    ).strip()

    if not exact_location:
        print("Location cannot be empty.")
        return

    # Search people
    results = search_people(exact_location)

    # Save raw response
    with open(
        "prospeo_quant_people.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            results,
            f,
            indent=4,
            ensure_ascii=False
        )

    print("\nSaved:")
    print("prospeo_quant_people.json")


def main():

    location = input(
        "Enter location (example: London): "
    ).strip()

    if not location:
        print("Location cannot be empty.")
        return

    # Get Prospeo location suggestions
    suggestions = find_location(location)

    if not suggestions:
        return

    location_suggestions = suggestions.get(
        "location_suggestions", []
    )

    if not location_suggestions:
        print("No location suggestions found.")
        return

    # Automatically select the first CITY result
    exact_location = location_suggestions[0]["name"]

    print("\nSelected location:")
    print(exact_location)

    # Search people
    results = search_people(exact_location)

    # Save raw response
    with open(
        "prospeo_quant_people.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            results,
            f,
            indent=4,
            ensure_ascii=False
        )

    print("\nSaved:")
    print("prospeo_quant_people.json")


if __name__ == "__main__":
    main()


Location search status: 200

Location suggestions:
{
  "error": false,
  "location_suggestions": [
    {
      "name": "London, United Kingdom",
      "type": "CITY"
    },
    {
      "name": "London, Ontario",
      "type": "CITY"
    },
    {
      "name": "London, Kentucky",
      "type": "CITY"
    },
    {
      "name": "London, Ohio",
      "type": "CITY"
    },
    {
      "name": "London, Michigan",
      "type": "CITY"
    },
    {
      "name": "London, Kansas",
      "type": "CITY"
    },
    {
      "name": "London, Arkansas",
      "type": "CITY"
    },
    {
      "name": "London, Minnesota",
      "type": "CITY"
    },
    {
      "name": "London, California",
      "type": "CITY"
    },
    {
      "name": "London, Texas",
      "type": "CITY"
    }
  ],
  "job_title_suggestions": null,
  "technology_suggestions": null,
  "industry_suggestions": null,
  "naics_suggestions": null,
  "sic_suggestions": null,
  "company_operating_languages_suggestions": null,
  "company_g